# Stage 09 — Homework Starter Notebook

In the lecture, we learned how to create engineered features. Now it’s your turn to apply those ideas to your own project data.

In [ ]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
# !pip install numpy
# !pip install pandas
# !pip install matplotlib

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Example synthetic data (replace with your project dataset)
np.random.seed(0)
n = 100
df = pd.DataFrame({
    'income': np.random.normal(60000, 15000, n).astype(int),
    'monthly_spend': np.random.normal(2000, 600, n).astype(int),
    'credit_score': np.random.normal(680, 50, n).astype(int),
    'region': np.random.choice(['North', 'South', 'East', 'West'], n),
    'default_flag': np.random.choice([0, 1], n, p=[0.8, 0.2]),
})
df.head()

,income,monthly_spend,credit_score,region,default_flag
0,86460,3129,661,East,0
1,66002,1191,668,North,0
2,74681,1237,734,West,0
3,93613,2581,712,South,1
4,88013,1296,712,East,1


## Engineered features — one is worked below; add at least two more

At least one of yours must encode a categorical column. `region` is the categorical
column here; the lecture shows three ways to encode it (one-hot, label, frequency).

In [11]:
# Worked example 1 of 3 — keep it, or swap in columns of your own.
df['spend_income_ratio'] = df['monthly_spend'] / df['income']
df[['monthly_spend', 'income', 'spend_income_ratio']].head()
# Write the rationale in the markdown cell below.

,monthly_spend,income,spend_income_ratio
0,3129,86460,0.036190
1,1191,66002,0.018045
2,1237,74681,0.016564
3,2581,93613,0.027571
4,1296,88013,0.014725


### Rationale for Feature 1
Explain why this feature may help a model. Reference your EDA.

The spend-to-income ratio measures spending relative to income rather than
looking at spending alone. This can help identify differences in spending
behavior between people with different income levels.

The Stage 08 EDA showed that income and spend are related, so this ratio may
provide additional information about the relative spending level.

In [3]:
# TODO: Add another feature
# Example: df['rolling_spend_mean'] = df['monthly_spend'].rolling(3).mean()
df['income_credit_ratio'] = df['income'] / df['credit_score']
df[['income', 'credit_score', 'income_credit_ratio']].head()

,income,credit_score,income_credit_ratio
0,86460,661,130.801815
1,66002,668,98.805389
2,74681,734,101.745232
3,93613,712,131.478933
4,88013,712,123.613764


### Rationale for Feature 2
Explain why this feature may help a model. Reference your EDA.

The income-to-credit-score ratio combines two important numerical variables.
It may help capture differences in financial capacity relative to credit quality.

This feature is motivated by the Stage 08 EDA, where the numerical variables
showed different scales and distributions. Combining them into a ratio may
provide a more comparable measure.

In [4]:
feature_corr_2 = df[['income_credit_ratio', 'default_flag']].corr()

print("Correlation with default_flag:")
print(feature_corr_2['default_flag'])

Correlation with default_flag:
income_credit_ratio    0.073348
default_flag           1.000000
Name: default_flag, dtype: float64


In [5]:
# TODO: Add a third feature. At least one of your three must ENCODE A CATEGORICAL
#   column - `region` is the one in this dataset.
#
#   The lecture shows three ways (section 'Categorical Encoding'):
#     one-hot     pd.get_dummies(df, columns=['region'])
#     label       LabelEncoder().fit_transform(df['region'])
#     frequency   df['region'].map(df['region'].value_counts(normalize=True))
#
#   Pick one and say WHY you picked it in the markdown cell below - the three are not
#   interchangeable, and that choice is the point of the exercise.
region_frequency = df['region'].value_counts(normalize=True)
df['region_frequency'] = df['region'].map(region_frequency)
df[['region', 'region_frequency']].head()

,region,region_frequency
0,East,0.28
1,North,0.22
2,West,0.22
3,South,0.28
4,East,0.28


### Rationale for Feature 3
Explain why this feature may help a model. Reference your EDA. If this is your
categorical encoding, say why you chose that encoding over the other two.

I use frequency encoding for the categorical variable `region`.

The Stage 08 EDA included a categorical profile of region, which showed the
distribution of observations across regions. Frequency encoding converts this
categorical information into a numerical feature that can be used directly
by a model.

I chose frequency encoding because it is simple and keeps the dataset compact,
while one-hot encoding would create several additional columns.

In [6]:
feature_corr_3 = df[['region_frequency', 'default_flag']].corr()

print("Correlation with default_flag:")
print(feature_corr_3['default_flag'])

Correlation with default_flag:
region_frequency    0.03307
default_flag        1.00000
Name: default_flag, dtype: float64


## Feature Correlation Summary

The following table compares the engineered features with the target variable.
Correlation is used as a simple screening tool rather than proof of causality.

In [12]:
features = [
    'spend_income_ratio',
    'income_credit_ratio',
    'region_frequency'
]

correlation_summary = df[features + ['default_flag']].corr()['default_flag'].drop('default_flag')

correlation_summary.sort_values(ascending=False)

income_credit_ratio    0.073348
region_frequency       0.033070
spend_income_ratio    -0.036101
Name: default_flag, dtype: float64

In [13]:
from pathlib import Path
import sys

ROOT = Path.cwd()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.features import (
    add_spend_income_ratio,
    add_income_credit_ratio,
    add_region_frequency
)

df_features = add_spend_income_ratio(df)
df_features = add_income_credit_ratio(df_features)
df_features = add_region_frequency(df_features)

df_features.head()

,income,monthly_spend,credit_score,region,default_flag,income_credit_ratio,region_frequency,spend_income_ratio
0,86460,3129,661,East,0,130.801815,0.28,0.036190
1,66002,1191,668,North,0,98.805389,0.22,0.018045
2,74681,1237,734,West,0,101.745232,0.22,0.016564
3,93613,2581,712,South,1,131.478933,0.28,0.027571
4,88013,1296,712,East,1,123.613764,0.28,0.014725
